# Diamond Price Analysis and Prediction

**Name:** Gauri Ghode 
**Roll Number:** SA207

## Problem Statement
This project analyzes the diamonds dataset to understand which factors affect diamond price and to build a simple model that predicts price. The original dataset has 53,940 rows and 10 columns, and after removing duplicate rows, 53794 rows remain for analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine.data import diamonds
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

This cell imports all libraries needed for data analysis, visualization, and machine learning in one place.

## Dataset Description
- Dataset Name: Diamonds Dataset
- Source: `plotnine` built-in dataset
- Original rows: 53,940
- Rows after duplicate removal: 53,794
- Columns: 10
- Main columns: carat, cut, color, clarity, depth, table, price, x, y, z

In [ ]:
df = diamonds.copy()
df.head()

The first five rows confirm that the dataset is loaded correctly and show the values stored in each column.

In [ ]:
print('Original shape:', df.shape)
print('\nData types:')
print(df.dtypes)

The data types show that the dataset contains both numeric and categorical columns.

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nDuplicate rows found:', df.duplicated().sum())
df = df.drop_duplicates()
print('Shape after removing duplicates:', df.shape)

No missing values were found. However, 146 duplicate rows were present, so they were removed before further analysis.

In [ ]:
stats = {}
for col in ['carat', 'price']:
    s = df[col]
    stats[col] = {
        'Mean': s.mean(),
        'Median': s.median(),
        'Mode': s.mode().iloc[0],
        'Standard Deviation': s.std(),
        'Variance': s.var(),
        'Range': s.max() - s.min(),
        'Mid-range': (s.max() + s.min()) / 2
    }

stats_df = pd.DataFrame(stats).round(2)
stats_df

These descriptive statistics summarize the center and spread of the cleaned `carat` and `price` columns.

### Data Cleaning Summary
The dataset had no missing values, so no filling or dropping was needed for nulls. It did contain 146 duplicate rows, and those duplicates were removed before generating charts and training the model.

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df['price'], bins=30)
plt.title('Histogram of Diamond Price')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

The histogram shows that most diamonds have lower prices, while a smaller number of diamonds have very high prices.

In [ ]:
cut_counts = df['cut'].value_counts()
plt.figure(figsize=(8,5))
plt.bar(cut_counts.index.astype(str), cut_counts.values)
plt.title('Count of Diamonds by Cut')
plt.xlabel('Cut')
plt.ylabel('Count')
plt.show()

The bar chart shows that Ideal cut diamonds are most common and Fair cut diamonds are least common.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df.assign(cut=df['cut'].astype(str)), x='cut', y='price', color='lightgray')
plt.title('Boxplot of Price by Cut')
plt.xlabel('Cut')
plt.ylabel('Price')
plt.show()

The boxplot shows price spread across cut categories and highlights many high-value outliers.

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='viridis', fmt='.2f')
plt.title('Correlation Heatmap of Numeric Columns')
plt.show()

The heatmap shows that carat has the strongest positive correlation with price.

## Simple Prediction

In [ ]:
X = df.drop(columns=['price'])
y = df['price']
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols)
])
pipeline = Pipeline([
    ('preprocess', preprocess),
    ('model', RandomForestRegressor(n_estimators=25, random_state=42, n_jobs=-1, max_depth=16))
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Mean Absolute Error:', round(mae, 2))
print('R2 Score:', round(r2, 4))

Categorical columns were encoded using one-hot encoding. The cleaned dataset was split into 80% training and 20% testing data. The model gave an R² score of 0.981, which means it explains most of the price variation, and the average prediction error is about 271.77 price units.

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(y_test, pred, alpha=0.25)
plt.title('Actual vs Predicted Diamond Price')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.show()

The prediction scatter plot shows that many predicted values are close to the actual values, so the model performs well overall.

## Insights and Recommendations

### Findings
1. The histogram shows a right-skewed price distribution, so cheaper diamonds are more common than expensive ones.
2. The cut bar chart shows that Ideal cut is the most common category in the cleaned dataset.
3. The boxplot shows many outliers, which means some diamonds are priced much higher than the rest.
4. The heatmap shows that carat has the strongest link with price.
5. The prediction model performed well because the R² score is high.

### Recommendations
1. Jewelers should focus strongly on carat and size measurements because they influence price a lot.
2. Shops can use a simple prediction model like this to estimate price quickly and support business decisions.